# Test III: Variational Quantum Classifier — PennyLane

**Task:** Classify gravitational lensing images into 3 dark matter substructure classes.

**Pipeline:**
```
Image (1, 150, 150) → Flatten → PCA (8 dims) → Angle Encoding (8 qubits)
    → VQC (3 layers, data re-uploading, parameter-shift gradient)
    → ⟨Z₀, Z₁, Z₂⟩ → Linear → 3 class logits
```

**Framework:** PennyLane + PyTorch (parameter-shift rule for exact gradients)

In [ ]:
import os, sys, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import pennylane as qml
import warnings; warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from model_vqc_pennylane import HybridVQC

print(f'PennyLane: {qml.__version__}')
print(f'PyTorch: {torch.__version__}')

## 1. Load Data & PCA

In [ ]:
CLASS_NAMES = ['no', 'sphere', 'vort']
DATA_DIR = 'data'
N_QUBITS = 8

def load_split(split):
    images, labels = [], []
    for ci, cn in enumerate(CLASS_NAMES):
        d = os.path.join(DATA_DIR, split, cn)
        for f in sorted(os.listdir(d)):
            if f.endswith('.npy'):
                images.append(np.load(os.path.join(d, f)).astype(np.float32).flatten())
                labels.append(ci)
    return np.array(images), np.array(labels)

print('Loading...')
X_train_raw, y_train = load_split('train')
X_val_raw, y_val = load_split('val')
print(f'Train: {X_train_raw.shape}, Val: {X_val_raw.shape}')

# PCA
pca = PCA(n_components=N_QUBITS)
X_train_pca = pca.fit_transform(X_train_raw)
X_val_pca = pca.transform(X_val_raw)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.4f}')

# Normalize to [0, 1]
pca_min, pca_max = X_train_pca.min(0), X_train_pca.max(0)
X_train_norm = (X_train_pca - pca_min) / (pca_max - pca_min + 1e-8)
X_val_norm = np.clip((X_val_pca - pca_min) / (pca_max - pca_min + 1e-8), 0, 1)

# Subsample
TRAIN_SUBSET = 3000
idx = []
for c in range(3):
    ci = np.where(y_train == c)[0]
    idx.extend(np.random.choice(ci, TRAIN_SUBSET // 3, replace=False).tolist())
np.random.shuffle(idx)
X_tr = torch.tensor(X_train_norm[idx], dtype=torch.float32)
y_tr = torch.tensor(y_train[idx], dtype=torch.long)
X_vl = torch.tensor(X_val_norm, dtype=torch.float32)
y_vl = torch.tensor(y_val, dtype=torch.long)
print(f'Train: {len(X_tr)}, Val: {len(X_vl)}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, N_QUBITS+1), pca.explained_variance_ratio_)
ax1.set_xlabel('Component'); ax1.set_ylabel('Variance Ratio')
ax1.set_title('PCA Explained Variance')
cumvar = np.cumsum(pca.explained_variance_ratio_)
ax2.plot(range(1, N_QUBITS+1), cumvar, 'bo-')
ax2.set_xlabel('Components'); ax2.set_ylabel('Cumulative')
ax2.set_title(f'Total: {cumvar[-1]*100:.1f}%')
ax2.axhline(y=0.9, color='r', ls='--', alpha=0.5)
plt.tight_layout(); plt.show()

## 2. Model — VQC with PennyLane

- 8 qubits, 3 variational layers with data re-uploading
- `Ry(feature × π + θ)` → ring CNOT → `Rz(θ)` per layer
- Readout: ⟨Z₀⟩, ⟨Z₁⟩, ⟨Z₂⟩ → Linear → 3 classes
- Gradient: **parameter-shift rule** (exact, PennyLane native)
- 48 quantum + 12 classical = **60 parameters**

In [ ]:
N_LAYERS = 3; N_CLASSES = 3
model = HybridVQC(n_qubits=N_QUBITS, n_layers=N_LAYERS, n_classes=N_CLASSES)
print(f'Parameters: {sum(p.numel() for p in model.parameters())}')
print(f'  Quantum: {model.q_weights.numel()}')
print(f'  Classical: {sum(p.numel() for p in model.post.parameters())}')

# Quick test
x_test = X_tr[:2]
out = model(x_test)
print(f'\nTest forward: {x_test.shape} → {out.shape}')
print(f'Output: {out.detach().numpy()}')

In [ ]:
# Benchmark
criterion = nn.CrossEntropyLoss()
t0 = time.time()
out = model(X_tr[:4])
loss = criterion(out, y_tr[:4])
loss.backward()
elapsed = time.time() - t0
print(f'Forward + backward (bs=4): {elapsed:.2f}s')
print(f'Estimated time/epoch (bs=16, 3000 samples): '
      f'{elapsed/4*16 * len(X_tr)/16 / 60:.1f} min')

## 3. Training

In [ ]:
BATCH_SIZE = 16; NUM_EPOCHS = 30; LR = 0.01

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_vl, y_vl), batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    model.train()
    rl, c, t = 0., 0, 0
    for X, y in train_loader:
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        rl += loss.item() * X.size(0)
        c += (out.argmax(1) == y).sum().item()
        t += X.size(0)

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            out = model(X)
            vc += (out.argmax(1) == y).sum().item()
            vt += X.size(0)

    scheduler.step()
    val_acc = vc / vt
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'weights/best_vqc_pennylane.pt')

    history['train_loss'].append(rl/t)
    history['train_acc'].append(c/t)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} ({time.time()-t0:.0f}s) | '
          f'Loss: {rl/t:.4f} Acc: {c/t:.4f} | '
          f'Val: {val_acc:.4f} | Best: {best_val_acc:.4f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], 'b-')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.grid(alpha=0.3)
ax2.plot(history['train_acc'], 'b-', label='Train')
ax2.plot(history['val_acc'], 'r-', label='Val')
ax2.axhline(y=1/3, color='gray', ls='--', alpha=0.5, label='Random')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'Accuracy (Best Val: {best_val_acc:.4f})')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Evaluation — ROC Curve & AUC

In [ ]:
model.load_state_dict(torch.load('weights/best_vqc_pennylane.pt'))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for X, y in val_loader:
        out = model(X)
        all_probs.append(torch.softmax(out, 1).numpy())
        all_labels.append(y.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_labels_bin = label_binarize(all_labels, classes=[0,1,2])

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cn} (AUC={auc(fpr,tpr):.4f})')

macro_auc = roc_auc_score(all_labels_bin, all_probs, multi_class='ovr', average='macro')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — PennyLane VQC ({N_QUBITS}q, {N_LAYERS}L) | AUC={macro_auc:.4f}')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/roc_vqc_pennylane.png', dpi=150)
plt.show()

print(f'Final Accuracy: {(all_probs.argmax(1)==all_labels).mean():.4f}')
for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    print(f'  {cn} AUC: {auc(fpr,tpr):.4f}')
print(f'Macro AUC: {macro_auc:.4f}')

## 5. Discussion

### Design Choices

| Choice | Detail |
|--------|--------|
| **Dimensionality reduction** | PCA: 22,500 → 8 (one feature per qubit) |
| **Encoding** | Angle encoding with data re-uploading (Ry) |
| **Circuit** | 8 qubits, 3 layers, ring CNOT, Ry+Rz per layer |
| **Readout** | ⟨Z₀⟩, ⟨Z₁⟩, ⟨Z₂⟩ → Linear → 3 logits |
| **Gradient** | Parameter-shift rule (exact, PennyLane native) |
| **Optimizer** | Adam (lr=0.01) + cosine annealing |
| **Parameters** | 48 quantum + 12 classical = 60 total |

### PennyLane vs CUDA-Q

| | PennyLane | CUDA-Q |
|---|---|---|
| Gradient | Parameter-shift (exact) | SPSA (noisy estimate) |
| PyTorch integration | Native autograd | Manual `torch.autograd.Function` |
| Hardware backends | IBM, IonQ, AWS | NVIDIA GPU simulator |
| Speed | Slower (2×N_params circuits/backward) | Faster (2 circuits/backward with SPSA) |
| Gradient quality | Exact | High variance |

### References
1. Pérez-Salinas et al., "Data re-uploading for a universal quantum classifier" (2020)
2. PennyLane: https://pennylane.ai/